In [1]:
import string
def preprocess_text(text, n):
    # 1. 转小写，去除标点，仅保留字母+空格
    text = text.lower()
    # 移除所有标点
    translator = str.maketrans('', '', string.punctuation)
    text_clean = text.translate(translator)
    # 2. 按空格分词
    tokens = text_clean.split()
    # 3. 构建词汇表：按词频排序，ID从0开始
    word_count = {}
    for word in tokens:
        word_count[word] = word_count.get(word, 0) + 1
    # 按频率降序排序
    sorted_words = sorted(word_count.keys(), key=lambda x: word_count[x], reverse=True)
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    # 4. 滑动窗口生成特征序列 + 标签
    features = []
    labels = []
    for i in range(len(tokens) - n):
        feat_seq = tokens[i:i+n]
        label = tokens[i+n]
        features.append(feat_seq)
        labels.append(label)
    return vocab, (features, labels)

# 测试
if __name__ == "__main__":
    test_text = "The time machine"
    vocab, (feats, labs) = preprocess_text(test_text, n=2)
    print("词汇表：", vocab)
    print("特征序列：", feats)
    print("标签序列：", labs)

词汇表： {'the': 0, 'time': 1, 'machine': 2}
特征序列： [['the', 'time']]
标签序列： ['machine']


In [2]:
import torch

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    前向传播计算单步隐藏状态
    x_t: (batch_size, input_size)
    h_prev: (batch_size, hidden_size)
    W_hx: (hidden_size, input_size)
    W_hh: (hidden_size, hidden_size)
    b_h: (hidden_size,)
    return: h_t, pre_act (激活前中间值，反向传播使用)
    """
    pre_act = torch.matmul(x_t, W_hx.t()) + torch.matmul(h_prev, W_hh.t()) + b_h
    h_t = torch.tanh(pre_act)
    return h_t, pre_act

def rnn_backward(dh_next, x_t, h_prev, pre_act, W_hx, W_hh):
    """
    反向传播，计算各梯度
    dh_next: 损失对h_t的上游梯度 (batch, hidden)
    pre_act: 激活前数值，用于tanh求导
    return: dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    # tanh导数: d(tanh(x)) = 1 - tanh(x)^2
    d_tanh = dh_next * (1 - torch.tanh(pre_act) ** 2)
    
    # 偏置梯度
    db_h = torch.sum(d_tanh, dim=0)
    # W_hx 梯度
    dW_hx = torch.matmul(d_tanh.t(), x_t)
    # W_hh 梯度
    dW_hh = torch.matmul(d_tanh.t(), h_prev)
    # 输入x_t梯度
    dx_t = torch.matmul(d_tanh, W_hx)
    # 上一隐状态梯度
    dh_prev = torch.matmul(d_tanh, W_hh)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试
if __name__ == "__main__":
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 初始化参数
    x_t = torch.randn(batch_size, input_size)
    h_prev = torch.randn(batch_size, hidden_size)
    W_hx = torch.randn(hidden_size, input_size)
    W_hh = torch.randn(hidden_size, hidden_size)
    b_h = torch.randn(hidden_size)
    
    # 前向
    h_t, pre_act = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("当前隐状态 h_t shape:", h_t.shape)
    
    # 模拟上游梯度
    dh_next = torch.randn_like(h_t)
    # 反向
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, x_t, h_prev, pre_act, W_hx, W_hh)
    print("dx_t shape:", dx_t.shape)
    print("dW_hx shape:", dW_hx.shape)
    print("dW_hh shape:", dW_hh.shape)
    print("db_h shape:", db_h.shape)

当前隐状态 h_t shape: torch.Size([2, 4])
dx_t shape: torch.Size([2, 3])
dW_hx shape: torch.Size([4, 3])
dW_hh shape: torch.Size([4, 4])
db_h shape: torch.Size([4])


In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,  # 双向RNN
            batch_first=False     # 输入格式: (seq_len, batch, input_dim)
        )
    
    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        return:
            all_h: 每个时间步拼接隐状态 (seq_len, batch, 2*hidden_dim)
            final_h: 最后时间步拼接隐状态 (batch, 2*hidden_dim)
        """
        # output: (seq_len, batch, 2*hidden_dim) 每个时间步输出
        # h_n: (2, batch, hidden_dim) 前向/后向最后隐状态
        output, h_n = self.rnn(X)
        all_h = output
        # 拼接前向、后向最终隐状态
        final_h = torch.cat([h_n[0], h_n[1]], dim=-1)
        return all_h, final_h

# 测试
if __name__ == "__main__":
    seq_len = 5
    batch = 2
    input_dim = 3
    hidden_dim = 4
    
    model = BiRNNEncoder(input_dim, hidden_dim)
    X = torch.randn(seq_len, batch, input_dim)
    all_h, final_h = model(X)
    print("每个时间步拼接隐状态 shape:", all_h.shape)
    print("最终拼接隐状态 shape:", final_h.shape)

每个时间步拼接隐状态 shape: torch.Size([5, 2, 8])
最终拼接隐状态 shape: torch.Size([2, 8])


In [4]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices, target_indices, W, W_out):
    """
    context_indices: 批次上下文索引列表 [batch, context_size]
    target_indices: 批次中心词索引 [batch]
    W: 嵌入权重 (V, d)
    W_out: 输出权重 (d, V)
    return: 平均交叉熵损失
    """
    batch_size = context_indices.shape[0]
    # 1. 查表得到上下文词向量
    context_emb = W[context_indices]  # (batch, context_size, d)
    # 2. 对上下文向量求平均
    avg_emb = torch.mean(context_emb, dim=1)  # (batch, d)
    # 3. 计算logits
    logits = torch.matmul(avg_emb, W_out)     # (batch, V)
    # 4. 交叉熵损失
    loss = F.cross_entropy(logits, target_indices)
    return loss

# 测试
if __name__ == "__main__":
    V = 10    # 词汇表大小
    d = 3     # 嵌入维度
    batch = 2
    context_size = 2
    
    # 初始化权重
    W = torch.randn(V, d)
    W_out = torch.randn(d, V)
    # 模拟输入
    context_idx = torch.tensor([[0, 1], [2, 3]])
    target_idx = torch.tensor([4, 5])
    
    loss = cbow_forward_loss(context_idx, target_idx, W, W_out)
    print("CBOW 交叉熵损失值:", loss.item())

CBOW 交叉熵损失值: 2.9606409072875977


In [5]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 单头维度
        
        # Q/K/V 投影层
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        # 输出线性层
        self.w_o = nn.Linear(d_model, d_model)
    
    def split_heads(self, x):
        """分头: (seq_len, batch, d_model) -> (num_heads, batch, seq_len, d_k)"""
        seq_len, batch, _ = x.shape
        x = x.view(seq_len, batch, self.num_heads, self.d_k)
        return x.permute(2, 1, 0, 3)
    
    def scaled_dot_product_attention(self, q, k, v):
        """缩放点积注意力"""
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weight = torch.softmax(attn_score, dim=-1)
        output = torch.matmul(attn_weight, v)
        return output
    
    def forward(self, X):
        """
        X: (seq_len, batch, d_model)
        return: 输出 (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape
        # 线性投影
        q = self.w_q(X)
        k = self.w_k(X)
        v = self.w_v(X)
        # 分头
        q = self.split_heads(q)
        k = self.split_heads(k)
        v = self.split_heads(v)
        # 单头注意力
        attn_out = self.scaled_dot_product_attention(q, k, v)
        # 拼接多头
        attn_out = attn_out.permute(2, 1, 0, 3).contiguous()
        attn_out = attn_out.view(seq_len, batch, self.d_model)
        # 最终线性层
        final_out = self.w_o(attn_out)
        return final_out

# 测试
if __name__ == "__main__":
    mha = MultiHeadAttention(d_model=4, num_heads=2)
    seq_len = 3
    batch = 2
    X = torch.randn(seq_len, batch, 4)
    out = mha(X)
    print("多头注意力输出 shape:", out.shape)

多头注意力输出 shape: torch.Size([3, 2, 4])
